In [1]:
import jams
import numpy as np
from pathlib import Path
from common import OUTPUT_DIM_NOTES
#pip install pyguitarpro
import guitarpro

# Define the directory containing JAMS files. The corresponding input audio files have identical names but with .wav extension. These names are deduced
# from the jams files automatically.
# jams_dirs=['/home/gerald/guitarset/annotation/rock']
jams_dirs='/data/SynthTab/Test-dev'#['/data/SynthTab/stratocaster_clean_bridge_jams']

output_data_dir='/data/data_slices/training'

def load_jams_file(file_path):
    with open(file_path, 'r') as f:
        jams_data = jams.load(f,validate=False)
    return jams_data
# intervals += (60 / tempo) * num_ticks / guitarpro.Duration.quarterTime
#Extract from synthtab jams file the note_tab annotations
def extract_guitar_annotations(jams_data,duration_bin_s=1000):
    #Get the tempi from the jams file
    tempos=[]
    for annotation in jams_data.annotations:
        if annotation.namespace=='tempo':
            for obs in annotation.data:
                tempos.append((obs.time,obs.value)) #(time,bpm)
    tempos=sorted(tempos,key=lambda x:x[0])
    
    annotations=[]
    for annotation in jams_data.annotations:
        if annotation.namespace=='note_tab':
            base_tuning=annotation.sandbox["open_tuning"]
            data=annotation.data
            for note in data:
                #Get the current tempo at the time of the note
                current_tempo=120
                for tempo in tempos:
                    if note.time>=tempo[0]:
                        current_tempo=tempo[1]
                #Convert from ticks to seconds
                ticks_per_quarter=guitarpro.Duration.quarterTime
                seconds_per_tick=60/(current_tempo*ticks_per_quarter)
                notetime=note.time*seconds_per_tick
                noteduration=note.duration*seconds_per_tick

                annotations.append({
                    'time':notetime,
                    'duration':noteduration,
                    'midi_note':note.value['fret']+base_tuning,
                    'confidence':note.confidence
                })
    return annotations
    # max_time=max(note['time']+note['duration'] for note in annotations)

    # num_bins=int(np.ceil(max_time/duration_bin_s))
    # binned_annotations=[[] for _ in range(num_bins)]
    # for note in annotations:
    #     bin_idx=int(note['time']//duration_bin_s)
    #     binned_annotations[bin_idx].append(note)
    # return binned_annotations,num_bins,max_time

def create_pianoroll(annotations, sr=48000, hop_length=256,  max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    max_time=max(note['time']+note['duration'] for note in annotations)
    frame_time=1/sr
    num_frames=int(np.ceil(max_time *sr))
    num_notes=max_note
    piano_roll=np.zeros((num_frames,num_notes))
    for note in annotations:
        start_frame=int(np.floor(note['time']*sr))
        end_frame=int(np.ceil((note['time']+note['duration'])*sr))
        note_idx=int(note['midi_note'])
        if note_idx>=0 and note_idx<num_notes:
            piano_roll[start_frame:end_frame,note_idx]=1
    piano_roll=np.swapaxes(piano_roll,0,1)
    onsets=np.zeros_like(piano_roll)
    onsets[:,1:]=np.diff(piano_roll,axis=1)
    onsets=np.max(onsets,axis=0)
    onsets[np.where(onsets<0)[0]]=1
    if onsets_2d:
        onsets=np.tile(onsets,(OUTPUT_DIM_NOTES,1))
    
    for t in range(piano_roll.shape[1]):
        notes=piano_roll[range(0,OUTPUT_DIM_NOTES),t].max()
        if notes==0:
            piano_roll[OUTPUT_DIM_NOTES-1,t]=1

    return piano_roll,onsets

def load_jams_annotations(file_path, sr=48000, hop_length=256, max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    jams_data=load_jams_file(file_path)
    return extract_guitar_annotations(jams_data)


I0000 00:00:1767908984.997765  134775 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1767908985.455292  134775 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1767908986.411005  134775 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
from fretboard import FretBoard
from common import frame_size,reshape_to_nn_input,reshape_to_nn_output,save_data_slices,plot_heatmap
from IPython.display import display
import os
from os import path
from scipy import io
import soundfile as sf
import matplotlib.pyplot as plt
import glob 
# Files where converted with for file in *.wav; do sox -SG "$file" -r 48000 -e float -b 32 48k/"$file"; done


def prepare_audio_midi_data(jams_file):

    # Compute the piano roll and onsets
    print(f"Processing {jams_file}")
    annotations=load_jams_annotations(jams_file)
    piano_roll,onsets=create_pianoroll(annotations,sr=48000,hop_length=frame_size,max_note=OUTPUT_DIM_NOTES,onsets_2d=False)
    # We want to cut out sections around the onsets
    # Compute the positions to preserve
    onset_positions=np.where(onsets>0)[0]
    for onset in onset_positions:
        onsets[(onset-4*frame_size):onset+4*frame_size]=1

    preserve_indices=np.where(onsets==0)[0]

    #Output directory
    # basename=os.path.basename(jams_file).replace('.jams','')
    jamsdirname=os.path.dirname(jams_file)
#get the parent directory
    jamsdirname=os.path.relpath(jamsdirname,start=jams_dirs)
    dirname=os.path.join(output_data_dir,jamsdirname)
    
    #Input audio
    audio_file=jams_file.replace('.jams','_48k.flac')#path.join(input_audio,basename+'_mix.wav')
    print("Loading audio file ",audio_file)
    audio,sample_rate=sf.read(audio_file)

    # Normalize the audio
    maxaudio=np.max(np.abs(audio))
    audio=audio/maxaudio
    plt.plot(audio)
    plt.show()

    # Check the sample rate
    if sample_rate!=48000:
        print("Unexpected sample rate ",sample_rate)
        return
    
    # Match piano roll length to audio length
    lenpianorool=piano_roll.shape[1]
    desired_length=len(audio)
    if lenpianorool<desired_length:
        # Pad with zeros
        padding = desired_length - lenpianorool
        piano_roll = np.pad(piano_roll, ((0, 0), (0, padding)), mode='constant')
    elif lenpianorool>desired_length:
        # Truncate
        piano_roll = piano_roll[:, :desired_length]

    # Compute the filterbank features
    filter =FretBoard(17.5,sample_rate)
    numfilters=filter.get_num_filters()
    filterbank_out=np.zeros((numfilters,len(audio)))
    filter.process(audio,filterbank_out)

    # Prune the onset features of the filterbank output and piano roll
    filterbank_out=np.take(filterbank_out,preserve_indices,axis=1,mode='clip')

    piano_roll=np.take(piano_roll,preserve_indices,axis=1,mode='clip')
    print("Sizes after onset pruning:"+str(filterbank_out.shape)+", "+str(piano_roll.shape))
    assert filterbank_out.shape[1]==piano_roll.shape[1],"Mismatched shapes after onset pruning"

    # Visualize the pruned data
    display(plot_heatmap(piano_roll))
    display(plot_heatmap(filterbank_out))

    # Reshape to NN input/output format
    input_slices=reshape_to_nn_input(filterbank_out)
    filterbank_out=[]
    output_slices=reshape_to_nn_output(piano_roll,frame_size)
    piano_roll=[]
    
    # Create output directories
    outdir=os.path.join(dirname,'output')
    indir=os.path.join(dirname,'input')
    
    
    Path(outdir).mkdir(parents=True,exist_ok=True)
    Path(indir).mkdir(parents=True,exist_ok=True)

    silent_range_step=frame_size//2
    ranges_to_preserve=np.zeros(output_slices.shape[0],dtype=bool)
    for i in range(0,output_slices.shape[0],silent_range_step):
        if np.sum(output_slices[i:(i+silent_range_step),(OUTPUT_DIM_NOTES-1)])==0:
            ranges_to_preserve[i:(i+silent_range_step)]=True
            # print(f"Marking  range for preservation: {i} to {i+silent_range_step}")

    percentage_preserved=100.0*np.sum(ranges_to_preserve)/ranges_to_preserve.shape[0]
    print(f"Preserving {percentage_preserved:.2f}% of data slices after removing silent ranges")

    input_slices=input_slices[ranges_to_preserve]
    output_slices=output_slices[ranges_to_preserve]
    # save the data slices
    print("Saving input data to ",indir)
    save_data_slices(indir,input_slices,1)
    
    print("Saving output data to ",outdir)
    
    save_data_slices(outdir,output_slices,1)

from joblib import Parallel, delayed

# Flatten files as shown above
all_jams_files = sorted(glob.glob(os.path.join(jams_dirs,'**', '*.jams'),recursive=True))# [f for d in jams_dirs for f in sorted(glob.glob(os.path.join(d,'**', '*.jams'),recursive=True))]

#Parallel(n_jobs=5)(delayed(prepare_audio_midi_data)(f) for f in all_jams_files) 

#print the tempo changes in a jams object
def print_tempo_changes(jams_data):
    for annotation in jams_data.annotations:
        if annotation.namespace=='tempo':
            data=annotation.data
            # get the number of tempo changes
            print(f"Number of tempo changes: {len(data)}")
            for tempo_change in data:
                #print(f"Time: {tempo_change.time}, Tempo: {tempo_change.value['qpm']}")
                print("tempo:",tempo_change)

                #print qpm from guitarpro
                guitarpro.Duration.quarterTime

                
 
for jams_file in all_jams_files:
    prepare_audio_midi_data(jams_file)
    print(f"Loading {jams_file}")
    jams_data=load_jams_file(jams_file)
    print_tempo_changes(jams_data)

    #Go from ticks to seconds, see https://github.com/yongyizang/SynthTab/blob/main/demo_embedding/SynthTab.py#L80
    
            

    


Processing /data/SynthTab/Test-dev/Scales and Arpeggios - Warmup - Marc Seal - gp3__1 - Electric Jazz Guitar__midi/stratocaster_clean_bridge_nonoise_stere.jams


In [ ]:
jamsdirname=os.path.dirname("/data/SynthTab/Test-dev/Vai, Steve - The Audience Is Listening - gp3__1 - Electric Clean Guitar__midi/stratocaster_clean_bridge_nonoise_stere.jams")
#get the parent directory
jamsdirname=os.path.relpath(jamsdirname,start='/data/SynthTab/Test-dev')

print(jamsdirname)